# 08 — TMDB Movie x Keyword Valence Join

Joins `tmdb_movies_clean` to `tmdb_keyword_lexicon` to produce two tables:

1. **Exploded** — 1 row per (movie, keyword): all lexicon columns attached to each keyword instance
2. **Movie aggregate** — 1 row per movie: keywords treated as a single ngram, computing
   a movie-level "computed valence" from the distribution of its keyword valences

**SCL filtering:** `exclude_from_scl = True` keywords are excluded from all valence
aggregation. These are metadata tags (place names, time periods, film form, genre labels,
source material, structural trivia, adult, identity/social) where SCL valence reflects
word connotation rather than narrative emotional content. See notebook 07 for full rationale.

## Movie-level aggregate columns

| column | description |
|--------|-------------|
| `total_keywords` | total keyword count for this movie |
| `excluded_keyword_count` | keywords removed by exclude_from_scl filter |
| `matched_keyword_count` | SCL-eligible keywords with a valence score |
| `keyword_match_rate` | `matched / (total - excluded)` (0.0-1.0) |
| `avg_keyword_valence` | mean valence across SCL-eligible matched keywords |
| `min/max_keyword_valence` | sentiment range |
| `valence_std` | spread — high std = tonally mixed movie |
| `in_scl_count` | SCL-eligible keywords with exact SCL phrase scores |
| `has_nma_count` | keywords with a negator/modal/adverb |
| `shift_strong_count` | keywords with `abs_composition_shift >= 0.30` |
| `avg_composition_shift` | mean deviation from naive word-averaging |
| `max_abs_composition_shift` | strongest single compositional surprise |
| `composition_type_dominant` | dominant linguistic mechanism (direct, opposing_polarity, idiomatic, ...) |


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA    = Path('data')
OUT_DIR = Path('output')

## Load data

In [2]:
movies = pd.read_csv(
    OUT_DIR / 'tmdb-movies-clean/tmdb_movies_clean.csv',
    low_memory=False,
    usecols=['id', 'title', 'release_date', 'vote_average', 'vote_count',
             'popularity', 'genres', 'keywords'],
)
movies = movies[movies['keywords'].notna() & (movies['keywords'] != '')]
print(f'Movies with keywords: {len(movies):,}')
movies.head(3)

Movies with keywords: 283,875


,id,title,vote_average,vote_count,release_date,popularity,genres,keywords
0,565770,Blue Beetle,7.139,1023,2023-08-16,2994.357,"Action, Science Fiction, Adventure","armor, superhero, family relationships, family..."
1,980489,Gran Turismo,8.068,702,2023-08-09,2680.593,"Action, Drama, Adventure","based on true story, racing, based on video ga..."
2,968051,The Nun II,6.545,365,2023-09-06,1692.778,"Horror, Mystery, Thriller","france, bullying, sequel, religion, demon, got..."


In [ ]:
lexicon = pd.read_csv(
    OUT_DIR / 'keyword-lexicon/tmdb_keyword_lexicon.csv',
    low_memory=False,
    usecols=['keyword', 'valence', 'valence_scale', 'valence_source',
             'in_scl', 'scl_token_coverage',
             'has_nma', 'composition_shift', 'abs_composition_shift',
             'shift_zscore', 'shift_percentile',
             'shift_strong', 'shift_very_strong', 'composition_type'],
)
print(f'Keyword lexicon: {len(lexicon):,} rows')
print(f'Coverage: {lexicon["valence"].notna().sum():,} keywords with valence ({lexicon["valence"].notna().mean()*100:.1f}%)')

# Merge keyword_type / is_adult / exclude_from_scl from nb 07 output
kw_types = pd.read_pickle(DATA / 'lexicons/tmdb_keyword_valence.pkl')[
    ['name', 'keyword_type', 'is_adult', 'exclude_from_scl']
].rename(columns={'name': 'keyword'})
lexicon = lexicon.merge(kw_types, on='keyword', how='left')
lexicon['keyword_type']    = lexicon['keyword_type'].fillna('other')
lexicon['is_adult']        = lexicon['is_adult'].fillna(False)
lexicon['exclude_from_scl'] = lexicon['exclude_from_scl'].fillna(False)

print(f'\nkeyword_type breakdown:')
print(lexicon['keyword_type'].value_counts().to_string())
print(f'\nexclude_from_scl: {lexicon["exclude_from_scl"].sum():,} ({lexicon["exclude_from_scl"].mean()*100:.1f}%)')
print(f'is_adult:          {lexicon["is_adult"].sum():,} ({lexicon["is_adult"].mean()*100:.1f}%)')


## Explode: 1 row per (movie, keyword)

All keywords are retained in the exploded output — `exclude_from_scl` is a column
you can filter on downstream, not a row filter applied here. The aggregate (below)
uses `exclude_from_scl` to decide which keywords contribute to valence stats, but
the full keyword set is always preserved for inspection, filtering, and non-valence uses.

In [4]:
# Parse comma-separated keywords → list
movies['keyword_list'] = movies['keywords'].str.split(',').apply(
    lambda ks: [k.strip().lower() for k in ks if k.strip()]
)

# Explode to 1 row per (movie, keyword)
exploded = (
    movies[['id', 'title', 'release_date', 'vote_average', 'vote_count',
             'popularity', 'genres', 'keyword_list']]
    .explode('keyword_list')
    .rename(columns={'keyword_list': 'keyword'})
    .reset_index(drop=True)
)

print(f'Exploded rows: {len(exploded):,}')
print(f'Unique movies:  {exploded["id"].nunique():,}')
print(f'Unique keywords: {exploded["keyword"].nunique():,}')

Exploded rows: 957,097
Unique movies:  283,773
Unique keywords: 63,725


In [ ]:
# Left join — all exploded rows are kept, unrecognised keywords get NaN lexicon columns
kw_joined = exploded.merge(lexicon, on='keyword', how='left')

n_total    = len(kw_joined)
n_excluded = kw_joined['exclude_from_scl'].fillna(False).sum()
n_eligible = n_total - n_excluded
n_matched  = kw_joined.loc[~kw_joined['exclude_from_scl'].fillna(False), 'valence'].notna().sum()

print(f'Total rows (all keywords kept):  {n_total:,}')
print(f'  exclude_from_scl = True:       {n_excluded:,}  ({n_excluded/n_total*100:.1f}%)')
print(f'  SCL-eligible:                  {n_eligible:,}  ({n_eligible/n_total*100:.1f}%)')
print(f'  SCL-eligible with valence:     {n_matched:,}  ({n_matched/n_eligible*100:.1f}% of eligible)')
print(f'\nkeyword_type breakdown across all rows:')
print(kw_joined['keyword_type'].fillna('other').value_counts().to_string())


## Aggregate: 1 row per movie

Valence statistics are computed **only over SCL-eligible keywords** (`exclude_from_scl = False`).
`total_keywords` still reflects every keyword the movie has — the exclusion only affects
what feeds the valence mean/min/max/std columns.

```
kw_joined (exploded, 957K rows)   ← ALL keywords retained
    │
    └── per movie groupby
            ├── total_keywords     = len(all keywords in group)
            ├── excluded_keyword_count = sum(exclude_from_scl)
            └── valence stats      = computed over exclude_from_scl == False only
```

In [ ]:
def agg_movie(g):
    n_total    = len(g)
    n_excluded = g['exclude_from_scl'].fillna(False).sum()

    # Only use SCL-eligible keywords (exclude_from_scl = False) for valence
    g_scl = g[~g['exclude_from_scl'].fillna(False)]
    matched = g_scl[g_scl['valence'].notna()]
    n_matched = len(matched)

    row = {
        'total_keywords':           n_total,
        'excluded_keyword_count':   n_excluded,
        'matched_keyword_count':    n_matched,
        'keyword_match_rate':       n_matched / (n_total - n_excluded) if (n_total - n_excluded) > 0 else 0.0,
    }

    if n_matched > 0:
        v = matched['valence']
        row['avg_keyword_valence']  = v.mean()
        row['min_keyword_valence']  = v.min()
        row['max_keyword_valence']  = v.max()
        row['valence_std']          = v.std(ddof=1) if n_matched > 1 else None
    else:
        row['avg_keyword_valence']  = None
        row['min_keyword_valence']  = None
        row['max_keyword_valence']  = None
        row['valence_std']          = None

    row['in_scl_count']       = g_scl['in_scl'].fillna(False).sum()
    row['has_nma_count']      = g_scl['has_nma'].fillna(False).sum()
    row['shift_strong_count'] = g_scl['shift_strong'].fillna(False).sum()

    scl_exact = g_scl[g_scl['in_scl'].fillna(False)]
    row['avg_composition_shift']     = scl_exact['composition_shift'].mean() if len(scl_exact) > 0 else None
    row['max_abs_composition_shift'] = scl_exact['abs_composition_shift'].max() if len(scl_exact) > 0 else None

    ct = g_scl['composition_type'].dropna()
    row['composition_type_dominant'] = ct.mode().iloc[0] if len(ct) > 0 else None

    return pd.Series(row)

print('Aggregating (valence computed over SCL-eligible keywords only)...')
agg = (
    kw_joined
    .groupby(['id', 'title', 'release_date', 'vote_average', 'vote_count', 'popularity', 'genres'])
    .apply(agg_movie, include_groups=False)
    .reset_index()
)

print(f'Movie aggregate: {agg.shape}')
print(f'\nexcluded_keyword_count stats:')
print(agg['excluded_keyword_count'].describe().round(2).to_string())
print(f'\navg_keyword_valence stats (SCL-eligible keywords only):')
print(agg['avg_keyword_valence'].describe().round(4).to_string())
print(f'\ncomposition_type_dominant breakdown:')
print(agg['composition_type_dominant'].value_counts(dropna=False).to_string())
agg.head(5)


## Examples

In [7]:
has_valence = agg[agg['avg_keyword_valence'].notna()]

print('Most positive keyword profiles (high avg_keyword_valence):')
print(has_valence.nlargest(8, 'avg_keyword_valence')
      [['title', 'release_date', 'avg_keyword_valence', 'matched_keyword_count', 'composition_type_dominant']]
      .to_string(index=False))

print()
print('Most negative keyword profiles:')
print(has_valence.nsmallest(8, 'avg_keyword_valence')
      [['title', 'release_date', 'avg_keyword_valence', 'matched_keyword_count', 'composition_type_dominant']]
      .to_string(index=False))

print()
print('Strongest composition shifts (movie keyword sets):')
print(agg.nlargest(8, 'max_abs_composition_shift')
      [['title', 'max_abs_composition_shift', 'avg_composition_shift', 'in_scl_count', 'composition_type_dominant']]
      .to_string(index=False))

Most positive keyword profiles (high avg_keyword_valence):
                        title release_date  avg_keyword_valence  matched_keyword_count composition_type_dominant
       Patisserie Coin De Rue   2011-02-11                  1.0                    1.0                    direct
              Drums of Tahiti   1954-04-23                  1.0                    1.0                    direct
                 Tiara Tahiti   1962-07-01                  1.0                    1.0                    direct
Une lubie de monsieur Fortune   2010-01-25                  1.0                    1.0                    direct
Curiosity, Adventure and Love   2016-07-01                  1.0                    1.0                    direct
             Go with Your Gut   2017-06-10                  1.0                    1.0                    direct
         Her Sound, Her Story   2018-05-11                  1.0                    1.0                       NaN
                     Paradise   2020-

## Save

In [ ]:
LEXICON_DIR = DATA / 'lexicons'
LEXICON_DIR.mkdir(parents=True, exist_ok=True)

# Pickles (internal use)
kw_out  = LEXICON_DIR / 'movie_keyword_exploded.pkl'
agg_out = LEXICON_DIR / 'movie_keyword_aggregate.pkl'
kw_joined.to_pickle(kw_out)
agg.to_pickle(agg_out)
print(f'Exploded  pkl → {kw_out}  ({kw_out.stat().st_size / 1e6:.1f} MB)  {kw_joined.shape}')
print(f'Aggregate pkl → {agg_out}  ({agg_out.stat().st_size / 1e6:.1f} MB)  {agg.shape}')

# CSV → output/movie-keyword-scores/ (Kaggle dataset)
scores_dir = Path('output/movie-keyword-scores')
scores_dir.mkdir(parents=True, exist_ok=True)
csv_out = scores_dir / 'movie_keyword_scores.csv'
agg.to_csv(csv_out, index=False)
print(f'\nAggregate csv → {csv_out}  ({csv_out.stat().st_size / 1e6:.1f} MB)')
print(f'Columns: {list(agg.columns)}')
